# 🦥 QwerySmith 1.1: Production Text-to-SQL Fine-Tuning Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyrax321/QwerySmith-1.0/blob/main/notebooks/QwerySmith_Colab_Training.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Cyrax321%2FQwerySmith--1.0-blue?logo=github)](https://github.com/Cyrax321/QwerySmith-1.0)
[![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97-Hugging%20Face-yellow)](https://huggingface.co/Cyrax321/QwerySmith-1.0)

This notebook runs the complete end-to-end **QwerySmith 1.1** pipeline on **Google Colab** (a free **Tesla T4 GPU** with ~15GB VRAM is fully supported):
- **Base Model**: `unsloth/Qwen3-4B`
- **Method**: QLoRA (Rank 16, Alpha 32, Dropout 0.05 via Unsloth FastLanguageModel)
- **Data Architecture**: Balanced multi-source training mix (`b-mc2/sql-create-context` + `gretelai/synthetic_text_to_sql`) with leak-proof evaluation splits and held-out cross-domain benchmarks (`sqale`, `large_schema`)
- **Stages**: Baselines (0-shot & 3-shot) $\rightarrow$ QLoRA Fine-Tuning $\rightarrow$ Strict In/Out-of-Distribution Evaluation $\rightarrow$ Statistical Comparison Report $\rightarrow$ 16-bit / GGUF Export & HF Hub Upload.

## 1. Verify GPU Allocation
Ensure that a GPU is assigned (`Runtime` $\rightarrow$ `Change runtime type` $\rightarrow$ `T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Set Directory
Pull the latest codebase from GitHub.

In [ ]:
import os
if not os.path.exists("QwerySmith-1.0"):
    !git clone https://github.com/Cyrax321/QwerySmith-1.0.git
%cd QwerySmith-1.0
!git pull origin main

## 3. Mount Google Drive (Persistent Storage)
Mount Google Drive so training checkpoints, adapter weights, evaluations, and GGUFs survive session disconnections.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/qwerysmith-1.1"
    print(f"Weights will be saved persistently to: {OUT_DIR}")
except Exception as e:
    OUT_DIR = "runs/qwerysmith-1.1"
    print(f"Saving locally to: {OUT_DIR}")
os.makedirs(OUT_DIR, exist_ok=True)

## 4. Install Dependencies
Install Unsloth, FlashAttention-compatible kernels, TRL, Datasets, and visual reporting tools.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets matplotlib tabulate

## 5. Step 1: Smoke Test (5–10 min Sanity Check)
Run a tiny end-to-end smoke test to verify model loading, tokenization, QLoRA patching, data loading, 30 steps of training, evaluation generation, and reporting before starting full training.

In [ ]:
!python QwerySmith/Qwerysmith_V11.py --smoke --out runs/qwerysmith-smoke

## 6. Step 2: Full Production Training (Run B Recipe)
Train QwerySmith 1.1 using the multi-source balanced mix:
- **Mix**: 5,000 examples from `sql_create_context` + 5,000 examples from `gretelai`
- **Held-Out Generalization Sets**: `sqale`, `large_schema`
- **Optimizations**: `lr=1e-4`, `dropout=0.05`, `batch_size=2`, `grad_accum=8` (effective batch size 16)

In [ ]:
!python QwerySmith/Qwerysmith_V11.py \
    --stage all \
    --mix sql_create_context:5000,gretel:5000 \
    --heldout sqale,large_schema \
    --epochs 1 \
    --lr 1e-4 \
    --dropout 0.05 \
    --batch-size 2 \
    --grad-accum 8 \
    --out "$OUT_DIR"

## 7. Step 3: Inspect Evaluation Results & Accuracy Chart
View the statistical comparison table (execution accuracy + 95% confidence intervals) across all test sets.

In [ ]:
import os
from IPython.display import Markdown, display, Image

results_file = os.path.join(OUT_DIR, "results.md")
chart_file = os.path.join(OUT_DIR, "comparison.png")

if os.path.exists(results_file):
    with open(results_file) as f:
        display(Markdown(f.read()))
else:
    print(f"Results file not found at: {results_file}")

if os.path.exists(chart_file):
    display(Image(chart_file))
else:
    print(f"Chart not found at: {chart_file}")

## 8. Step 4: Export Model (16-bit, GGUF) & Push to Hugging Face Hub
Login with your Hugging Face write token and export the model formats:

In [ ]:
from huggingface_hub import login
# Paste your Hugging Face token (Write permissions)
login()

# Export standalone 16-bit model, 4-bit GGUF, and upload adapter
!python QwerySmith/Qwerysmith_V11.py \
    --stage export \
    --merge \
    --gguf \
    --push Cyrax321/QwerySmith-1.1 \
    --out "$OUT_DIR"

## 9. Step 5: Test the Self-Healing SQL Agent
Run queries through the self-healing execution loop with schema introspection and runtime recovery.

In [ ]:
!python agent.py --query "Find the average salary of employees in engineering department"